<a href="https://colab.research.google.com/github/mohamedelguendy/information-system-security-project/blob/main/information_system_security_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ast
import re

# Definitions for detection
DANGEROUS_MODULES = ['pickle', 'eval', 'exec', 'os.system', 'input', 'subprocess']
UNSAFE_FUNCTIONS = {
    'os': ['system', 'popen', 'spawnl', 'spawnle', 'spawnlp', 'spawnlpe', 'spawnv', 'spawnve', 'spawnvp', 'spawnvpe'],
    'subprocess': ['run', 'Popen', 'call', 'check_call', 'check_output'],
    'pickle': ['load', 'loads', 'dump', 'dumps'],
    'hashlib': ['md5', 'sha1'],
    'builtin': ['eval', 'exec', 'open']
}

def is_increment(node):
    """Return +1 if node represents increment (e.g. x += 1 or x = x + 1),
    -1 if decrement, 0 otherwise."""
    # AugAssign: x += 1 or x -= 1
    if isinstance(node, ast.AugAssign) and isinstance(node.target, ast.Name):
        if isinstance(node.op, ast.Add):
            if isinstance(node.value, ast.Constant) and isinstance(node.value.value, (int, float)):
                return +1 if node.value.value > 0 else 0
        elif isinstance(node.op, ast.Sub):
            if isinstance(node.value, ast.Constant) and isinstance(node.value.value, (int, float)):
                return -1 if node.value.value > 0 else 0

    # Assign: x = x + 1 or x = x - 1
    if isinstance(node, ast.Assign):
        if (len(node.targets) == 1 and isinstance(node.targets[0], ast.Name) and
            isinstance(node.value, ast.BinOp) and isinstance(node.value.left, ast.Name) and
            node.value.left.id == node.targets[0].id):
            if isinstance(node.value.op, ast.Add):
                if isinstance(node.value.right, ast.Constant) and isinstance(node.value.right.value, (int, float)):
                    return +1 if node.value.right.value > 0 else 0
            elif isinstance(node.value.op, ast.Sub):
                if isinstance(node.value.right, ast.Constant) and isinstance(node.value.right.value, (int, float)):
                    return -1 if node.value.right.value > 0 else 0
    return 0

# endless loop detection
def detect_endless_loop(code):
    try:
        tree = ast.parse(code)

        for node in ast.walk(tree):
            if isinstance(node, ast.While):
                # Detect while True (classic endless loop)
                if isinstance(node.test, ast.Constant) and node.test.value is True:
                    return "Potential endless loop (while True)"

                # For simple conditions like: x < constant, x <= constant, x > constant, x >= constant
                if isinstance(node.test, ast.Compare) and len(node.test.ops) == 1 and len(node.test.comparators) == 1:
                    left = node.test.left
                    op = node.test.ops[0]
                    right = node.test.comparators[0]

                    # We only handle simple variable comparisons like x < 5
                    if isinstance(left, ast.Name) and isinstance(right, ast.Constant) and isinstance(right.value, (int, float)):
                        var_name = left.id
                        comp_val = right.value

                        # Find how var_name is modified inside the loop
                        increments = []

                        for body_node in node.body:
                            for sub_node in ast.walk(body_node):
                                inc = is_increment(sub_node)
                                # We only consider increments/decrements of var_name
                                target_name = None
                                if isinstance(sub_node, ast.AugAssign) and isinstance(sub_node.target, ast.Name):
                                    target_name = sub_node.target.id
                                elif isinstance(sub_node, ast.Assign) and len(sub_node.targets) == 1 and isinstance(sub_node.targets[0], ast.Name):
                                    target_name = sub_node.targets[0].id

                                if inc != 0 and target_name == var_name:
                                    increments.append(inc)

                        if not increments:
                            return f"Potential endless loop (condition variable '{var_name}' not modified)"

                        # Check direction of increments relative to condition
                        # x < 5 or x <= 5: x must increase to eventually break condition
                        # x > 5 or x >= 5: x must decrease to eventually break condition

                        # Sum all increments/decrements to get net direction
                        net_inc = sum(increments)

                        # Detect endless loop if net_inc moves variable away from terminating condition
                        if isinstance(op, (ast.Lt, ast.LtE)) and net_inc <= 0:
                            return f"Potential endless loop (variable '{var_name}' in condition '{ast.dump(op)}' modified in wrong direction)"
                        if isinstance(op, (ast.Gt, ast.GtE)) and net_inc >= 0:
                            return f"Potential endless loop (variable '{var_name}' in condition '{ast.dump(op)}' modified in wrong direction)"

                # Fallback: check if variables in condition are modified at all
                vars_in_condition = {n.id for n in ast.walk(node.test) if isinstance(n, ast.Name)}

                vars_modified = set()
                for body_node in node.body:
                    for sub_node in ast.walk(body_node):
                        if isinstance(sub_node, ast.Assign):
                            for target in sub_node.targets:
                                if isinstance(target, ast.Name):
                                    vars_modified.add(target.id)
                        elif isinstance(sub_node, ast.AugAssign):
                            if isinstance(sub_node.target, ast.Name):
                                vars_modified.add(sub_node.target.id)

                unmodified = vars_in_condition - vars_modified
                if vars_in_condition and unmodified == vars_in_condition:
                    return f"Potential endless loop (condition variables not modified: {', '.join(unmodified)})"

        return "No endless loop found"
    except Exception as e:
        return f"Loop analysis error: {e}"

# String-based module/function detection
def detect_insecure_modules(text):
    found = [mod for mod in DANGEROUS_MODULES if mod in text]
    return "Insecure modules/functions used: " + ", ".join(found) if found else "No insecure modules found"

# AST-based unsafe function detection
def detect_ast_unsafe_functions(code):
    try:
        tree = ast.parse(code)
        warnings = []
        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                func = node.func
                if isinstance(func, ast.Name):
                    if func.id in UNSAFE_FUNCTIONS.get('builtin', []):
                        warnings.append(f"Use of built-in unsafe function: {func.id}")
                elif isinstance(func, ast.Attribute) and isinstance(func.value, ast.Name):
                    module = func.value.id
                    function = func.attr
                    if module in UNSAFE_FUNCTIONS and function in UNSAFE_FUNCTIONS[module]:
                        warnings.append(f"Unsafe function: {module}.{function}")
        return "\n".join(warnings) if warnings else "No unsafe functions found (AST)"
    except Exception as e:
        return f"AST parsing error: {e}"

# Enforced password policy
def check_password_strength(password):
    errors = []
    if len(password) < 8:
        errors.append("Password must be at least 8 characters")
    if not re.search(r'[A-Za-z]', password):
        errors.append("Password must include at least one letter")
    if not re.search(r'\d', password):
        errors.append("Password must include at least one number")
    if not re.search(r'\W', password):
        errors.append("Password must include at least one special character")

    if errors:
        print("Weak password:\n- " + "\n- ".join(errors))
        return False
    else:
        print("Password is strong")
        return True

# Main interactive program with multiline input support
def main():
    print("Enter Python code to scan and (unsafely) evaluate (finish input with an empty line):")
    lines = []
    while True:
        line = input()
        if line.strip() == "":
            break
        lines.append(line)
    user_input = "\n".join(lines)

    print("\n=== Analyzing code before execution ===")
    loop_check = detect_endless_loop(user_input)
    print(loop_check)

    if "Potential endless loop" in loop_check:
        print("Execution skipped due to potential endless loop detected.")
    else:
        print("\n=== Executing with exec (UNSAFE) ===")
        try:
            exec(user_input)
            print("Code executed successfully.")
        except Exception as e:
            print("Execution failed:", e)

    print("\n=== Scan Results ===")
    print(detect_insecure_modules(user_input))
    print(detect_ast_unsafe_functions(user_input))

    # Enforce password strength rules until passed
    while True:
        password = input("\nEnter a password to check strength: ")
        if check_password_strength(password):
            break

if __name__ == "__main__":
    main()

Enter Python code to scan and (unsafely) evaluate (finish input with an empty line):
while True:     print("This runs forever")
f


=== Analyzing code before execution ===
Potential endless loop (while True)
Execution skipped due to potential endless loop detected.

=== Scan Results ===
No insecure modules found
No unsafe functions found (AST)

Enter a password to check strength: ;'iyhaseriophyq2389r89
Password is strong


In [ ]:
#It protects code execution areas, like:
#Code playgrounds
#Online editorsAdmin panels with scripts
#“Run JS” buttons
#Sandboxed interpreters